# Скоринг МСБ: шанс одобрения, честная ставка и «что если»

Демо на **19 600 реальных юрлицах** из открытых данных ФНС и ГИР БО (bo.nalog.gov.ru).

| Что | Откуда |
|---|---|
| Оборот, возраст, ОКВЭД, долг/выручка, долг/EBIT, капитал/активы, DSCR, рост выручки, налоговый долг | **реальные** (ФНС opendata, бухотчётность ГИР БО) |
| Просрочки в БКИ, сезонность выручки, продукт, сумма, РКО, маркетплейс, гарантия, страховка | **синтетика** - в открытом доступе нет |
| Факт дефолта | **синтетика** - скрытая логика с шумом |
| Фондирование | **реальная** G-кривая Мосбиржи на D-1 |

Три модели, связанные в одну цепочку:

1. **Модель 1a - PD** (риск дефолта): скоркарта OptBinning, 8–12 факторов.
2. **Модель 1b - шанс одобрения** (политика банка): логистическая регрессия на one-hot категориях, баллы по категориям.
3. **Модель 2 - ставка**: `ставка = фондирование + PD×LGD + капитал + расходы + маржа − скидки`, плюс ПСК с комиссией и страховкой.

Сверху - what-if («добавьте залог - ставка −X п.п.») и маршрутизация вместо тупикового отказа.

In [ ]:
import json, os, sys, warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("SCORING_DATA_DIR", ROOT / "data"))
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160, "display.max_columns", 30)

from scoring.gcurve import CurveStore
from scoring import synth
from scoring.whatif import Engine, evaluate, whatif, SCENARIOS
from scoring.approval import CATEGORIES, LABELS

# Оформление графиков: светлая поверхность, тонкая сетка, текст - нейтральными чернилами, серии - по порядку
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID,
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 1, "axes.axisbelow": True,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
                     "legend.frameon": False})

## Данные

In [ ]:
path = DATA / "scoring" / "train.csv"
if path.exists():
    df = pd.read_csv(path, dtype={"inn": str, "okved": str}, encoding="utf-8-sig")
    SRC = "ФНС + ГИР БО + синтетика"
else:  # без загрузки ФНС - полностью синтетическая выборка той же схемы
    df = synth.generate(20000, seed=7, asof=date.today())
    SRC = "синтетика"
curves_path = DATA / "raw" / "gcurve_params.csv"
curves = CurveStore.from_csv(curves_path) if curves_path.exists() else None
ASOF = date(2026, 10, 22) if SRC != "синтетика" else date.today()
print(f"{SRC}: {len(df):,} компаний, доля дефолтов {df.default_12m.mean():.1%}; "
      f"G-кривая {'до ' + curves.last_date.isoformat() if curves else 'нет - запасная ставка'}")
df[["inn", "okved_section", "revenue", "age_months", "debt_to_revenue", "dscr", "tax_debt_to_revenue",
    "product", "requested_amount", "requested_term_m", "bki_max_dpd_12m", "default_12m"]].head()

## Обучение всей цепочки

PD-скоркарта → кредитная логика со ставкой модели 2 в DSCR-лимите → модель одобрения на её решениях.

In [ ]:
engine, fit = Engine.fit(df, curves=curves, asof=ASOF, seed=7)
rep, out = fit["scorecard"], fit["assessed"]
cv = rep["cv"]
print(f"PD: Gini по 5 фолдам {cv.gini_test.mean():.3f} ± {cv.gini_test.std(ddof=0):.3f}, "
      f"KS {rep['ks_test']:.3f}, факторов {len(rep['factors'])}")
print("Факторы:", ", ".join(rep["factors"]))
ar = fit["approval"]
print(f"Одобрение: {fit['approval_n']:,} решений, доля одобренных {fit['approval_rate']:.1%}; "
      f"Gini по 5 фолдам {np.mean(ar['cv_gini_test']):.3f} ± {np.std(ar['cv_gini_test']):.3f} "
      f"(только категории заявки - {np.mean(ar['cv_gini_test_categories']):.3f})")

## Модель 1a: PD-скоркарта

Шкала: 600 баллов при шансах 50:1, +20 баллов - шансы вдвое лучше. Класс риска 1…7 по PD. Ниже - откалиброван ли PD:
средний предсказанный PD против наблюдаемой доли дефолтов на отложенной выборке.

In [ ]:
cls = rep["classes"]
fig, ax = plt.subplots(figsize=(8, 3.6))
x, w = np.arange(len(cls)), 0.38
ax.bar(x - w / 2, 100 * cls.pd_mean, w, color=SERIES[0], edgecolor=SURFACE, linewidth=2, label="предсказанный PD")
ax.bar(x + w / 2, 100 * cls.default_rate, w, color=SERIES[1], edgecolor=SURFACE, linewidth=2, label="наблюдаемая доля дефолтов")
ax.set_xticks(x, [f"класс {c}" for c in cls.risk_class])
ax.set_ylabel("%")
ax.set_title("PD по классам риска, отложенная выборка", loc="left", color=INK)
for i, v in enumerate(100 * cls.default_rate):
    ax.annotate(f"{v:.1f}", (x[i] + w / 2, v), ha="center", va="bottom", fontsize=8, color=INK2)
ax.legend(loc="upper left")
plt.show()
cls.assign(pd_mean=(100 * cls.pd_mean).round(2), default_rate=(100 * cls.default_rate).round(2))

In [ ]:
rep["iv"].round(3)  # IV всех кандидатов; selected - вошёл в скоркарту

## Модель 1b: шанс одобрения

`logit(p) = β0 + Σ β·x`, все признаки - категории с одной **референсной** (её коэффициент 0, остальные - эффект относительно неё).
Перевод в баллы: `балл = 600 + (20/ln2)·logit(p)`, `баллы категории = (20/ln2)·β`.

Модель учится на решениях нашей кредитной логики - то есть это модель **политики**, а не риска (риск - модель 1a).
Стоп-факторы применяются до неё: при стоп-факторе шанс 0 и показывается причина.

In [ ]:
fit["approval_table"].round(2)

In [ ]:
expected = {"product=express": "−", "business_age=24-36": "−", "rko=да": "+", "bank_turnover=>100%": "+",
            "amount_share=>30%": "−", "marketplace=да": "+", "guarantee=да": "+", "bki_overdue=>30 дн": "−",
            "logit_pd": "−", "cash_cover": "+", "ln_turnover_to_payment": "+"}
rows = []
for k, sign in expected.items():
    b = engine.approval.coef[k]
    got = "0" if round(b, 3) == 0 else ("+" if b > 0 else "−")
    rows.append({"признак": k, "ожидали": sign, "получили": got, "beta": round(b, 3),
                 "не против смысла": got in (sign, "0")})
pd.DataFrame(rows)

Знаки ограничены смыслом, поэтому «против смысла» не бывает: где данные тянули в другую сторону, эффект равен нулю
(«ограничение знака» в таблице баллов). Нули - не ошибка: например, просрочки и возраст уже учтены через PD, и
отдельного вклада в решение у них нет. Модель одобрения показывает, *как банк решает*, а не *почему клиент надёжен*.

## Модель 2: ставка

`ставка = фондирование + EL + капитал + расходы + маржа − скидки`, `EL = PD × LGD × (1 − покрытие гарантии)`.

- фондирование - доходность G-кривой Мосбиржи на среднюю жизнь кредита по кривой на D-1 + спред;
- LGD по продукту: залоговый 35%, экспресс 70%, карта 75% …; гарантия Корпорации МСП покрывает 50% потерь;
- капитал = вес риска по ПВР × норматив 12% × (стоимость капитала − фондирование). Вес риска - по Положению
  Банка России 845-П: заявка до 70 млн руб. - розничное МСП (собственный LGD, но не ниже 30%), крупнее -
  корпоративное требование базового ПВР (LGD 40%, залог недвижимости - 20% на обеспеченную часть, M = 2,5 года,
  скидка к корреляции для малых компаний). Вес растёт с PD: надёжному клиенту капитал дешевле;
- скидки: обороты в банке ≥ 40% / > 100% лимита, подписка, страховка.

**ПСК** (честная ставка) - IRR месячных потоков с комиссией за выдачу и страховкой. Все параметры иллюстративные (`config/pricing.json`).

In [ ]:
q = out[out.pricing.notna()]
parts = pd.DataFrame(list(q.pricing), index=q.index).assign(risk_class=q.risk_class)
cols = ["funding", "el", "capital", "opex", "margin"]
names = {"funding": "фондирование", "el": "PD×LGD", "capital": "капитал", "opex": "расходы", "margin": "маржа",
         "discounts": "скидки"}
by = parts.groupby("risk_class")[cols + ["discounts", "rate", "all_in", "risk_weight"]].mean() * 100
fig, ax = plt.subplots(figsize=(8, 4))
bottom = np.zeros(len(by))
for i, c in enumerate(cols):
    ax.bar(by.index, by[c], bottom=bottom, color=SERIES[i], edgecolor=SURFACE, linewidth=2, label=names[c])
    bottom += by[c].to_numpy()
ax.bar(by.index, -by["discounts"], color=SERIES[5], edgecolor=SURFACE, linewidth=2, label="скидки (вычитаются)")
ax.scatter(by.index, by["all_in"], color=INK, s=36, zorder=3, label="ПСК")
for k, r in by.iterrows():
    ax.annotate(f"ставка {r['rate']:.1f}\nПСК {r['all_in']:.1f}", (k, r["all_in"]), xytext=(0, 8),
                textcoords="offset points", ha="center", va="bottom", fontsize=8, color=INK2)
ax.set_ylim(top=by["all_in"].max() * 1.18)
ax.axhline(0, color=INK2, linewidth=1)
ax.set_xlabel("класс риска"); ax.set_ylabel("% годовых")
ax.set_title("Из чего складывается ставка по классам риска", loc="left", color=INK)
ax.legend(ncol=4, loc="upper left", fontsize=8, bbox_to_anchor=(0, -0.15))
plt.show()
by.round(2)

In [ ]:
prod = (out.assign(предложение=out.outcome.isin(["APPROVE", "COUNTER"]))
        .groupby("product").agg(заявок=("outcome", "size"), доля_предложений=("предложение", "mean"),
                                шанс_одобрения=("approval_probability", "mean"),
                                ставка=("rate", "mean"), ПСК=("all_in", "mean")))
(prod * [1, 100, 100, 100, 100]).round(1)

## Решения и маршрутизация вместо отказа

In [ ]:
print(out.outcome.value_counts(normalize=True).round(3).to_dict())
out[out.route.notna()].apply(lambda r: f"{r['product']} → {r.offered_product}", axis=1).value_counts()

## What-if для одной компании

**Шанс одобрения** - оценка модели 1b *до подачи заявки* по категориям заявки, PD и запасу денежного потока: её можно показать клиенту заранее.
**Решение** - результат полной проверки (отчётность, DSCR, стоп-факторы) после подачи. Поэтому шанс 34% и решение
«одобрено» не противоречат друг другу: шанс - средний исход для похожих заявок.

Берём реальную компанию, которая просит экспресс-кредит без залога, и смотрим, что изменится, если она добавит залог,
переведёт обороты в банк, получит гарантию МСП и т. д. Обе модели и ставка пересчитываются вместе.

In [ ]:
pick = out[(out["product"] == "express") & (out.outcome.isin(["APPROVE", "COUNTER"])) & (out.risk_class.between(3, 4))]
app = df.loc[pick.index[0]].to_dict()
res = whatif(engine, app)
b = res["base"]
print(f"ИНН {b['inn']}: PD {b['pd']:.2%}, класс {b['risk_class']}, шанс одобрения {b['approval_probability']:.0%}")
print(f"Решение {b['outcome']}: {b['offered_product']} {b['offered_amount']:,.0f} руб. из {b['requested_amount']:,.0f}; "
      f"ставка {b['pricing']['rate']:.2%}, ПСК {b['pricing']['all_in']:.2%}")
print("Вклад в шанс одобрения:", ", ".join(f"{x['признак']} {x['категория']} {x['баллы']:+.0f}" for x in b["approval_explain"]))
wf = pd.DataFrame(res["scenarios"])[["title", "outcome", "offered_product", "approval_probability", "delta_approval_pp",
                                     "rate", "delta_rate_pp", "all_in", "delta_all_in_pp"]]
wf.round(3)

In [ ]:
s = wf.dropna(subset=["delta_rate_pp"])
fig, axes = plt.subplots(1, 3, figsize=(12, 0.6 + 0.45 * len(s)), sharey=True)
for ax, col, title in [(axes[0], "delta_approval_pp", "Шанс одобрения, п.п."), (axes[1], "delta_rate_pp", "Ставка, п.п."),
                       (axes[2], "delta_all_in_pp", "ПСК (честная ставка), п.п.")]:
    ax.barh(s.title, s[col], color=SERIES[0], edgecolor=SURFACE, linewidth=2, height=0.6)
    ax.axvline(0, color=INK2, linewidth=1)
    ax.set_title(title, loc="left", color=INK)
    for y, v in enumerate(s[col]):
        ax.annotate(f"{v:+.1f}", (v, y), xytext=(4 if v >= 0 else -4, 0), textcoords="offset points",
                    ha="left" if v >= 0 else "right", va="center", fontsize=8, color=INK2)
    ax.grid(axis="y", visible=False)
    lim = max(abs(s[col]).max(), 0.5) * 1.45
    ax.set_xlim(-lim, lim)
axes[0].invert_yaxis()
plt.tight_layout(); plt.show()

## Нет тупиковых отказов

Компания без расчётного счёта просит овердрафт - продукт ей недоступен, но вместо отказа она получает альтернативу с причиной.

In [ ]:
young = {**app, "product": "overdraft", "has_rko": 0, "bank_turnover_ratio": 0.0}
r = evaluate(engine, young)
print(f"Просила: {r['product']} → предложено: {r['offered_product'] or 'ничего'} на {r['offered_amount']:,.0f} руб. ({r['outcome']})")
if r["route"]:
    print("Почему не подошёл запрошенный продукт:", "; ".join(x["category"] for x in r["route"]["reasons"]))

## Ответ в JSON (например, для n8n)

In [ ]:
print(json.dumps(evaluate(engine, app), ensure_ascii=False, indent=1)[:2500])

## Ограничения - честно

- **Дефолты, БКИ, сезонность и поля заявки - синтетика**: механизм и интерфейс настоящие, абсолютные PD и ставки иллюстративные.
- **Одобрение ≠ дефолт.** Модель 1b учит политику, модель 1a - риск; показываем их раздельно.
- **Reject inference.** На реальных данных дефолт виден только у выданных кредитов - PD нужно достраивать по отказникам (reject inference).
- **Выборка - компании со сданной отчётностью**: моложе 9 месяцев в ней нет, поэтому правило «бизнес < 9 мес.» работает как стоп-правило продукта, а не как коэффициент.